<a href="https://colab.research.google.com/github/ThanhB18059162022/MMRec/blob/dev/preprocessing/1splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 基于rating2inter.ipynb生成的5-core交互图，Train/Validation/Test data splitting
- Based on generated interactions, perform data splitting


In [1]:
import os, csv
import pandas as pd

In [2]:
# os.chdir('/home/enoche/MMRec/Sports14')
# os.getcwd()

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PATH = "/content/drive/MyDrive/Classroom/HTTT_Projects/CD HTTT TM/MMRec/data"

In [5]:
!cp "{PATH}/sports14-indexed.inter" "."

## 直接加载现成的, Load interactions

In [6]:
rslt_file = 'sports14-indexed.inter'
df = pd.read_csv(rslt_file, sep='\t')
print(f'shape: {df.shape}')
df[:4]

shape: (296337, 5)


,userID,itemID,rating,timestamp,x_label
0,0,0,5.0,1390694400,0
1,1,0,5.0,1328140800,0
2,2,0,4.0,1330387200,0
3,3,0,4.0,1328400000,0


In [7]:
import random
import numpy as np

In [8]:
df = df.sample(frac=1).reset_index(drop=True)

df.sort_values(by=['userID'], inplace=True)
df[:20]

,userID,itemID,rating,timestamp,x_label
256694,0,15852,5.0,1390694400,0
95809,0,13372,5.0,1391990400,1
51705,0,3369,5.0,1405123200,2
240139,0,17787,3.0,1391990400,1
1424,0,5458,5.0,1405123200,2
225566,0,11981,2.0,1390694400,0
114536,0,3327,3.0,1391990400,1
19505,0,0,5.0,1390694400,0
229111,1,9198,5.0,1318377600,0
4926,1,1542,4.0,1302220800,0


In [9]:
uid_field, iid_field = 'userID', 'itemID'

uid_freq = df.groupby(uid_field)[iid_field]
u_i_dict = {}
for u, u_ls in uid_freq:
    u_i_dict[u] = list(u_ls)
list(u_i_dict.items())[:3]

[(0, [15852, 13372, 3369, 17787, 5458, 11981, 3327, 0]),
 (1,
  [9198,
   1542,
   7169,
   13468,
   11502,
   0,
   14212,
   15278,
   2374,
   4123,
   6677,
   2322,
   8802,
   7215,
   15249,
   3087,
   4281,
   5044,
   6554]),
 (2,
  [9841,
   10218,
   6445,
   2950,
   8776,
   500,
   3011,
   14657,
   6298,
   7950,
   0,
   1114,
   9250,
   14212,
   11360,
   9254,
   14699,
   15360,
   3661,
   4155,
   15075,
   14242,
   10837])]

In [11]:
list(u_i_dict.keys())[:3]

[0, 1, 2]

In [15]:
new_label = []
u_ids_sorted = sorted(u_i_dict.keys())

for u in u_ids_sorted:
    items = u_i_dict[u]
    # get num interact
    n_items = len(items)
    if n_items < 10:
        # take 1 for test 1 for val and rest for train
        tmp_ls = [0] * (n_items - 2) + [1] + [2]
    else:
        # split 80% train, 10% val, 10% test
        val_test_len = int(n_items * 0.2)
        train_len = n_items - val_test_len
        val_len = val_test_len // 2
        test_len = val_test_len - val_len
        tmp_ls = [0] * train_len + [1] * val_len + [2] * test_len
    new_label.extend(tmp_ls)

new_label[:50]

[0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 2,
 2]

In [17]:
df['x_label'] = new_label
df[:20]

,userID,itemID,rating,timestamp,x_label
256694,0,15852,5.0,1390694400,0
95809,0,13372,5.0,1391990400,0
51705,0,3369,5.0,1405123200,0
240139,0,17787,3.0,1391990400,0
1424,0,5458,5.0,1405123200,0
225566,0,11981,2.0,1390694400,0
114536,0,3327,3.0,1391990400,1
19505,0,0,5.0,1390694400,2
229111,1,9198,5.0,1318377600,0
4926,1,1542,4.0,1302220800,0


In [18]:
rslt_file[:-6]

'sports14-indexed'

In [20]:
new_labeled_file = rslt_file[:-6] + '-v4.inter'
df.to_csv(os.path.join('./', new_labeled_file), sep='\t', index=False)
print('done!!!')

done!!!


In [21]:
!cp "{new_labeled_file}" "{PATH}/{new_labeled_file}"

## Reload

In [22]:
indexed_df = pd.read_csv(new_labeled_file, sep='\t')
print(f'shape: {indexed_df.shape}')
indexed_df[:20]

shape: (296337, 5)


,userID,itemID,rating,timestamp,x_label
0,0,15852,5.0,1390694400,0
1,0,13372,5.0,1391990400,0
2,0,3369,5.0,1405123200,0
3,0,17787,3.0,1391990400,0
4,0,5458,5.0,1405123200,0
5,0,11981,2.0,1390694400,0
6,0,3327,3.0,1391990400,1
7,0,0,5.0,1390694400,2
8,1,9198,5.0,1318377600,0
9,1,1542,4.0,1302220800,0


In [24]:
u_id_str, i_id_str = 'userID', 'itemID'
u_uni = indexed_df[u_id_str].unique()
c_uni = indexed_df[i_id_str].unique()

print(f'# of unique learners: {len(u_uni)}')
print(f'# of unique courses: {len(c_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(u_uni), max(u_uni)))
print('min/max of unique courses: {0}/{1}'.format(min(c_uni), max(c_uni)))


# of unique learners: 35598
# of unique courses: 18357
min/max of unique learners: 0/35597
min/max of unique courses: 0/18356
